# 028 — Speculative RAG
**Advanced RAG Series** | Google 2024: small-LM drafts + large-LM verifies. Cuts latency ~50%.

Covers: Parallel drafting · Document partitioning · Draft selection · Latency comparison


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    return open(path, encoding='utf-8').read()

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
dl_text  = load_doc("deep_learning.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars")


✓ Loaded 22,486 chars


In [4]:
# ── 1. What is Speculative RAG? ─────────────────────────────────────────
print("STANDARD RAG:")
print("  retrieve N docs → concatenate all → large LLM reads all → answer")
print("  Problem: large LLM must read a huge context. Slow. Expensive.")
print()
print("SPECULATIVE RAG (Google 2024):")
print("  1. Partition retrieved docs into K subsets (one per cluster)")
print("  2. SMALL LLM generates K draft answers in parallel, one per subset")
print("  3. LARGE LLM reads the K drafts (tiny! not the full docs) + selects/synthesizes")
print()
print("Why it's faster:")
print("  Large LLM reads K short drafts instead of the full retrieved context.")
print("  Drafting (small LLM) and large LLM processing overlap in production pipelines.")
print("  Paper: 12.97% accuracy improvement + 51% latency reduction on PubHealth.")


STANDARD RAG:
  retrieve N docs → concatenate all → large LLM reads all → answer
  Problem: large LLM must read a huge context. Slow. Expensive.

SPECULATIVE RAG (Google 2024):
  1. Partition retrieved docs into K subsets (one per cluster)
  2. SMALL LLM generates K draft answers in parallel, one per subset
  3. LARGE LLM reads the K drafts (tiny! not the full docs) + selects/synthesizes

Why it's faster:
  Large LLM reads K short drafts instead of the full retrieved context.
  Drafting (small LLM) and large LLM processing overlap in production pipelines.
  Paper: 12.97% accuracy improvement + 51% latency reduction on PubHealth.


In [5]:
# ── 2. Setup retrieval ──────────────────────────────────────────────────
import os, time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
docs_lc = splitter.create_documents([rag_text, ml_text, ai_text])

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(docs_lc, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
print(f"✓ Vector store: {vectorstore.index.ntotal} vectors, retrieves top-8")


✓ Vector store: 65 vectors, retrieves top-8


In [6]:
# ── 3. Partition retrieved docs into subsets ─────────────────────────────
import numpy as np
from sklearn.cluster import KMeans

def partition_docs(docs, n_subsets: int = 4):
    '''Cluster docs by embedding similarity → n_subsets groups.'''
    doc_vecs = np.array(embeddings.embed_documents([d.page_content for d in docs]))
    n_clusters = min(n_subsets, len(docs))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(doc_vecs)
    subsets = {}
    for doc, label in zip(docs, labels):
        subsets.setdefault(label, []).append(doc)
    return list(subsets.values())

question = "How does retrieval augmented generation improve LLM accuracy?"
retrieved_docs = retriever.invoke(question)
subsets = partition_docs(retrieved_docs, n_subsets=4)
print(f"Retrieved {len(retrieved_docs)} docs → {len(subsets)} subsets")
for i, subset in enumerate(subsets):
    print(f"  Subset {i}: {len(subset)} docs")


Retrieved 8 docs → 4 subsets
  Subset 0: 2 docs
  Subset 1: 2 docs
  Subset 2: 3 docs
  Subset 3: 1 docs


In [7]:
# ── 4. Small LLM drafts one answer per subset ───────────────────────────
DRAFT_PROMPT = (
    "Read the context below and write a concise draft answer (2-3 sentences).\n"
    "If the context is insufficient, say what is missing.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Draft answer:"
)

def generate_draft(subset, question):
    context = "\n\n".join([d.page_content for d in subset])
    return llm.invoke(DRAFT_PROMPT.format(context=context, question=question)).content

print("Generating drafts with small LLM (llama-3.1-8b)...")
t0 = time.time()
drafts = []
for i, subset in enumerate(subsets):
    draft = generate_draft(subset, question)
    drafts.append(draft)
    print(f"  Draft {i}: {draft[:120]}...")
draft_time = time.time() - t0
print(f"\n✓ Drafts generated in {draft_time:.1f}s")


Generating drafts with small LLM (llama-3.1-8b)...


  Draft 0: Retrieval-Augmented Generation (RAG) improves Large Language Model (LLM) accuracy by leveraging external knowledge bases...


  Draft 1: Unfortunately, the context provided does not contain enough information to answer the question about how retrieval augme...


  Draft 2: Unfortunately, the provided context does not mention retrieval augmented generation. To answer the question, more inform...


  Draft 3: Based on the provided context, it seems that retrieval augmented generation is not explicitly mentioned. However, I can ...

✓ Drafts generated in 1.9s


In [8]:
# ── 5. Large LLM selects and synthesizes from drafts ────────────────────
VERIFY_PROMPT = (
    "You are given {n} draft answers to a question, each written from a different subset of documents.\n"
    "Select the most accurate and complete drafts, then synthesize a final answer.\n"
    "Be concise. Do not add information not in the drafts.\n\n"
    "Question: {question}\n\n"
    "Drafts:\n{drafts_text}\n\n"
    "Final synthesized answer:"
)

drafts_text = "\n\n".join([f"Draft {i+1}:\n{d}" for i, d in enumerate(drafts)])
t1 = time.time()
final_answer = llm_smart.invoke(
    VERIFY_PROMPT.format(n=len(drafts), question=question, drafts_text=drafts_text)
).content
verify_time = time.time() - t1

print("FINAL ANSWER (large LLM synthesis):")
print("=" * 50)
print(final_answer)
print(f"\nTiming: drafts={draft_time:.1f}s  verify={verify_time:.1f}s  total={draft_time+verify_time:.1f}s")


FINAL ANSWER (large LLM synthesis):
Retrieval-Augmented Generation (RAG) improves Large Language Model (LLM) accuracy by leveraging external knowledge bases to retrieve relevant documents, which are then used as context for generating responses. This approach allows RAG systems to tap into a vast amount of information not stored in the model's parameters, leading to more accurate and informative responses. By combining the strengths of dense and sparse retrieval methods, RAG can generate more accurate and relevant responses. Additionally, RAG reduces the likelihood of errors and improves overall accuracy by informing and refining the LLM's response with external knowledge and context from a cache or database.

Timing: drafts=1.9s  verify=0.4s  total=2.3s


In [9]:
# ── 6. Compare: standard RAG vs speculative RAG ─────────────────────────
# Standard: large LLM reads all retrieved docs directly
STD_PROMPT = (
    "Answer the question using the context below.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\nAnswer:"
)

std_context = "\n\n".join([d.page_content for d in retrieved_docs])
t2 = time.time()
std_answer = llm_smart.invoke(
    STD_PROMPT.format(context=std_context, question=question)
).content
std_time = time.time() - t2

print("STANDARD RAG (large LLM reads all docs directly):")
print(std_answer)
print(f"\nStandard RAG time  : {std_time:.1f}s  (large LLM reads {len(std_context):,} chars)")
print(f"Speculative RAG time: {draft_time + verify_time:.1f}s  (large LLM reads {len(drafts_text):,} chars of drafts)")
print(f"Context reduction   : {(1 - len(drafts_text)/len(std_context))*100:.0f}%")


STANDARD RAG (large LLM reads all docs directly):
Retrieval-Augmented Generation (RAG) improves Large Language Model (LLM) accuracy by combining information retrieval with text generation. Rather than relying solely on knowledge stored in model parameters, RAG systems retrieve relevant documents from an external knowledge base and use them as context for generating responses. This approach allows the model to access a broader range of information and provide more accurate and up-to-date responses. The use of external knowledge bases and retrieval mechanisms, such as dense and sparse retrieval, enables RAG to address knowledge gaps and limitations in the LLM, resulting in more accurate and informative responses. Additionally, RAG's ability to evaluate its own performance using metrics such as faithfulness, answer relevancy, context recall, and context precision helps to further improve its accuracy.

Standard RAG time  : 0.7s  (large LLM reads 2,126 chars)
Speculative RAG time: 2.3s  (l

In [10]:
# ── 7. Summary ──────────────────────────────────────────────────────────
print("SPECULATIVE RAG — KEY POINTS")
print("=" * 45)
print()
print("Core idea:")
print("  Inspired by speculative decoding in LLM inference.")
print("  Small model drafts many candidates cheaply.")
print("  Large model verifies/selects efficiently (reads short drafts, not full docs).")
print()
print("Partitioning strategy:")
print("  Cluster retrieved docs by embedding similarity (KMeans).")
print("  One doc per cluster in the strict Google version.")
print("  Each subset = a different 'perspective' on the question.")
print()
print("Production benefit:")
print("  In a real system, N draft calls run in PARALLEL.")
print("  Large LLM verification only reads K short drafts.")
print("  End-to-end latency = max(draft_latencies) + verify_latency")
print("  vs standard = verify_latency on full context (much larger).")
print()
print("Paper: Speculative RAG (Google, 2024) — arxiv.org/abs/2407.08223")


SPECULATIVE RAG — KEY POINTS

Core idea:
  Inspired by speculative decoding in LLM inference.
  Small model drafts many candidates cheaply.
  Large model verifies/selects efficiently (reads short drafts, not full docs).

Partitioning strategy:
  Cluster retrieved docs by embedding similarity (KMeans).
  One doc per cluster in the strict Google version.
  Each subset = a different 'perspective' on the question.

Production benefit:
  In a real system, N draft calls run in PARALLEL.
  Large LLM verification only reads K short drafts.
  End-to-end latency = max(draft_latencies) + verify_latency
  vs standard = verify_latency on full context (much larger).

Paper: Speculative RAG (Google, 2024) — arxiv.org/abs/2407.08223
